**Library**

In [1]:
install.packages("fda")
install.packages("tvReg")
install.packages("readxl")
install.packages("tidyverse")
install.packages("dplyr")
install.packages("ggplot2")
install.packages("patchwork")
install.packages("writexl")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘mvtnorm’, ‘locfit’, ‘ash’, ‘FNN’, ‘kernlab’, ‘mclust’, ‘multicool’, ‘pracma’, ‘pcaPP’, ‘hdrcde’, ‘colorspace’, ‘ks’, ‘bitops’, ‘rainbow’, ‘RCurl’, ‘fds’, ‘deSolve’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘fracdiff’, ‘timeDate’, ‘cowplot’, ‘Deriv’, ‘forecast’, ‘microbenchmark’, ‘numDeriv’, ‘doBy’, ‘SparseM’, ‘MatrixModels’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘carData’, ‘abind’, ‘pbkrtest’, ‘quantreg’, ‘lme4’, ‘miscTools’, ‘rbibutils’, ‘car’, ‘lmtest’, ‘sandwich’, ‘strucchange’, ‘urca’, ‘RcppArmadillo’, ‘bdsmatrix’, ‘collapse’, ‘zoo’, ‘maxLik’, ‘Rdpack’, ‘Formula’, ‘systemfit’, ‘vars’, ‘bvarsv’, ‘plm’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package 

In [2]:
library(fda)
library(tvReg)
library(readxl)
library(fda)
library(dplyr)
library(tidyverse)
library(ggplot2)
library(patchwork)
library(writexl)

Loading required package: splines

Loading required package: fds

Loading required package: rainbow

Loading required package: MASS

Loading required package: pcaPP

Loading required package: RCurl

Loading required package: deSolve


Attaching package: ‘fda’


The following object is masked from ‘package:graphics’:

    matplot


The following object is masked from ‘package:datasets’:

    gait


Loading required package: Matrix

Funded by the Horizon 2020. Framework Programme of the European Union.



Attaching package: ‘dplyr’


The following object is masked from ‘package:MASS’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.2.0
✔ ggplot2   4.0.3     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.2     ✔ 

**Data**

In [3]:
data <- read_excel("DATA_LN3.xlsx")
head(data)

No,Tanggal,Waktu,Y,Hari,LN_NS,LN_M,LN_NM,NEWLN
<dbl>,<dttm>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,2020-01-01,00.30,3602.309,Rabu,1,0,0,1
2,2020-01-01,01.00,3525.583,Rabu,1,0,0,1
3,2020-01-01,01.30,3465.264,Rabu,1,0,0,1
4,2020-01-01,02.00,3400.974,Rabu,1,0,0,1
5,2020-01-01,02.30,3368.715,Rabu,1,0,0,1
6,2020-01-01,03.00,3326.610,Rabu,1,0,0,1


**Modeling**

In [ ]:
Y <- data$Y[721:87648]
Ylag1 <- data$Y[673:87600]
Ylag2 <- data$Y[625:87552]
Ylag3 <- data$Y[577:87504]
Ylag4 <- data$Y[529:87456]
Ylag5 <- data$Y[481:87408]
Ylag6 <- data$Y[433:87360]
Ylag7 <- data$Y[385:87312]
Ylag8 <- data$Y[337:87264]
Ylag9 <- data$Y[289:87216]
Ylag10 <- data$Y[241:87168]
Ylag11 <- data$Y[193:87120]
Ylag12 <- data$Y[145:87072]
Ylag13 <- data$Y[97:87024]
Ylag14 <- data$Y[49:86976]
Ylag15 <- data$Y[1:86928]

#Dummy LN
LN <- data$NEWLN[721:87648]

#Dummy LN 3 type
NS <- data$LN_NS[721:87648]
M <- data$LN_M[721:87648]
NM <- data$LN_NM[721:87648]

n_day <- 1811
intervals_per_day <- 48
n <- n_day * intervals_per_day

# Slot waktu dalam sehari (1 sampai 48), diulang untuk 30 hari
slot <- rep(1:intervals_per_day, n_day)
slot_scaled <- (slot - 1) / (intervals_per_day - 1)

# --- 3 spesifikasi dummy ---
formulas <- list(
  TanpaDummy = Y ~ 1 + Ylag1 + Ylag7 + Ylag8 + Ylag14 + Ylag15,
  DummyLN    = Y ~ 1 + Ylag1 + Ylag7 + Ylag8 + Ylag14 + Ylag15 + LN,
  Dummy3Type = Y ~ 1 + Ylag1 + Ylag7 + Ylag8 + Ylag14 + Ylag15 + NS + M + NM
)

bws <- c(0.043, 0.086, 0.128)
ker <- "Triweight"

# --- Grid 9 model. bw bervariasi paling cepat di dalam tiap spesifikasi dummy ---
configs <- expand.grid(bw = bws, spec = names(formulas),
                       KEEP.OUT.ATTRS = FALSE, stringsAsFactors = FALSE)

# --- Grid 48 titik intrahari ---
n_points    <- 48
slot_id     <- 1:n_points
time_scaled <- (slot_id - 1) / (n_points - 1)

# --- Wadah hasil ---
results <- vector("list", nrow(configs))
models  <- vector("list", nrow(configs))

# --- Loop 9 model: fit -> simpan 48 titik -> summary ---
for (i in seq_len(nrow(configs))) {
  spec_i <- configs$spec[i]
  bw_i   <- configs$bw[i]
  cat("\n>>> Menjalankan Model", i, "dari", nrow(configs),
      "|", spec_i, "| bw =", bw_i, "...\n"); flush.console()

  model_i <- tvLM(formulas[[spec_i]],
                  z = slot_scaled, bw = bw_i, tkernel = ker, est = "ll")

  coef_mat <- model_i$coefficients[1:n_points, , drop = FALSE]   # 48 slot
  results[[i]] <- data.frame(
    Slot        = slot_id,
    Time_scaled = time_scaled,
    coef_mat,
    check.names = FALSE
  )
  models[[i]] <- model_i
  names(results)[i] <- paste0("M", i, "_", spec_i, "_bw", bw_i)

  cat("\n===== Summary Model", i, "|", spec_i, "| bw =", bw_i, "=====\n")
  print(summary(model_i)); flush.console()
}

# --- Ekspor 9 sheet ---
write_xlsx(results, path = "Estimasi_FRTVCM_Dummy9Model.xlsx")


>>> Menjalankan Model 1 dari 9 | TanpaDummy | bw = 0.043 ...

===== Summary Model 1 | TanpaDummy | bw = 0.043 =====

Call: 
tvLM(formula = formulas[[spec_i]], z = slot_scaled, bw = bw_i, 
    est = "ll", tkernel = ker)

Class:  tvlm 

Summary of time-varying estimated coefficients: 
        (Intercept)  Ylag1  Ylag7   Ylag8 Ylag14  Ylag15
Min.          166.4 0.6422 0.4165 -0.4130 0.3761 -0.2912
1st Qu.       207.8 0.6570 0.4269 -0.3965 0.3848 -0.2716
Median        278.6 0.7557 0.4367 -0.3794 0.3899 -0.2591
Mean          284.8 0.7401 0.4452 -0.3788 0.3903 -0.2530
3rd Qu.       369.9 0.8088 0.4656 -0.3588 0.3958 -0.2304
Max.          401.2 0.8421 0.4722 -0.3434 0.4021 -0.2183

Bandwidth:  0.043
Pseudo R-squared:  0.8683 


Class:  tvlm 

Mean of coefficient estimates: 
(Intercept)       Ylag1       Ylag7       Ylag8      Ylag14      Ylag15 
   284.7678      0.7401      0.4452     -0.3788      0.3903     -0.2530 

Bandwidth:  0.043 


>>> Menjalankan Model 2 dari 9 | TanpaDummy | bw = 0.

In [ ]:
# slot + tanggal
residual_df <- data.frame(
  Slot = slot,
  Tanggal = data$Tanggal[721:87648])

# tambah residual 9 model
for(i in seq_along(models)){
  residual_df[[paste0("Residual_M", i)]] <-
    models[[i]]$residuals}

# export 1 sheet
write_xlsx(
  list(Residual = residual_df),
  path = "Residual_9Model_Triweight.xlsx")

**Diagnostic**

In [4]:
resid_all <- read_excel("Residual_9Model_Triweight.xlsx")
head(resid_all)

n_day <- 1811
intervals_per_day <- 48
n <- n_day * intervals_per_day

# Slot waktu dalam sehari (1 sampai 48), diulang untuk 30 hari
slot <- rep(1:intervals_per_day, n_day)
slot_scaled <- (slot - 1) / (intervals_per_day - 1)

Slot,Tanggal,Residual_M1,Residual_M2,Residual_M3,Residual_M4,Residual_M5,Residual_M6,Residual_M7,Residual_M8,Residual_M9
<dbl>,<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2020-01-16,87.18917,86.44670,86.46222,102.46677,101.79082,101.78025,107.2499,106.71852,106.7816
2,2020-01-16,123.73112,124.12792,124.25943,138.39217,138.76767,138.88679,143.6781,143.91984,144.0127
3,2020-01-16,119.33842,119.09120,119.00459,135.15791,134.98467,134.93032,140.9565,140.72705,140.6754
4,2020-01-16,116.54207,116.18386,115.77868,132.89780,132.63594,132.28221,138.9906,138.91270,138.5498
5,2020-01-16,131.11109,130.03577,129.97744,146.33817,145.26312,145.09368,152.1048,151.03799,150.6697
6,2020-01-16,63.59292,63.56195,64.62937,80.26596,80.07381,80.54719,87.0300,86.53577,86.4648


In [5]:
# ===== resid test =====
residTest <- function(scores, eigen.resid, eigen.covariate, bin, M = 1000) {
  set.seed(123)
  K <- scores %>% distinct(k) %>% pull()
  n <- scores %>% filter(k == 1) %>% count() %>% pull()
  J <- names(scores %>% dplyr::select(-k, -resid))

  Tk <- c()
  for (i in K) {
    Tj <- c()
    for (j in J) {
      tj <- scores %>%
        dplyr::select(k, resid) %>%
        mutate(covariate = as.matrix(scores[j])) %>%
        filter(k == i) %>%
        mutate_if(is.matrix, as.numeric) %>%
        arrange(covariate) %>%
        mutate(bin = bin) %>%
        group_by(bin) %>%
        summarize(mean = mean(resid)) %>%
        summarize(var = var(mean))

      Tj <- c(Tj, tj$var)
    }
    Tk <- c(Tk, weighted.mean(Tj, eigen.covariate))
  }

  T0 <- weighted.mean(Tk, eigen.resid)

  T0m <- c()
  for (j in 1:M) {
    Tm <- c()
    for (i in K) {
      tm <- scores %>%
        filter(k == i) %>%
        dplyr::select(resid) %>%
        sample_n(n) %>%
        mutate(bin = bin) %>%
        group_by(bin) %>%
        summarize(mean = mean(resid)) %>%
        summarize(var = var(mean))

      Tm <- c(Tm, tm$var)
    }
    T0m <- c(T0m, weighted.mean(Tm, eigen.resid))
  }

  pvalue <- mean(T0m >= T0)

  list(stat = T0, dist = T0m, pvalue = pvalue)
}

In [6]:
# ===== SETUP KONSTANTA =====

t   <- slot_scaled
day <- data$Tanggal[721:87648]

BSpline <- function(nbasis) create.bspline.basis(c(1, 48), nbasis)

bins_list <- list(
  bin1 = c(rep(1:15, each = 120), rep(16, each = 11)),
  bin2 = c(rep(1:30, each = 60),  rep(31, each = 11)),
  bin3 = c(rep(1:60, each = 30),  rep(61, each = 11)))

# ===== DEFINISI KOVARIAT PER SPESIFIKASI =====

covariates_by_spec <- list(

  TanpaDummy = list(
    Ylag1  = data$Y[673:87600],
    Ylag7  = data$Y[385:87312],
    Ylag8  = data$Y[337:87264],
    Ylag14 = data$Y[49:86976],
    Ylag15 = data$Y[1:86928]),

  DummyLN = list(
    Ylag1  = data$Y[673:87600],
    Ylag7  = data$Y[385:87312],
    Ylag8  = data$Y[337:87264],
    Ylag14 = data$Y[49:86976],
    Ylag15 = data$Y[1:86928],
    LN     = data$NEWLN[721:87648]),

  Dummy3Type = list(
    Ylag1  = data$Y[673:87600],
    Ylag7  = data$Y[385:87312],
    Ylag8  = data$Y[337:87264],
    Ylag14 = data$Y[49:86976],
    Ylag15 = data$Y[1:86928],
    NS     = data$LN_NS[721:87648],
    M      = data$LN_M[721:87648],
    NM     = data$LN_NM[721:87648]))

# ===== LABEL 9 MODEL =====

bws   <- c(0.043, 0.086, 0.128)
specs <- c("TanpaDummy", "DummyLN", "Dummy3Type")

configs <- expand.grid(bw   = bws,
                       spec = specs,
                       KEEP.OUT.ATTRS = FALSE,
                       stringsAsFactors = FALSE)

# ===== PRA-HITUNG PCA KOVARIAT PER SPESIFIKASI =====

pca_covariates <- list()

for (spec in specs) {
  cat("\n>>> PCA kovariat untuk spesifikasi:", spec, "\n")
  pca_covariates[[spec]] <- list()
  cov_list <- covariates_by_spec[[spec]]

  for (cov_name in names(cov_list)) {
    cat("  Memproses kovariat:", cov_name, "...\n"); flush.console()

    cov_df  <- data.frame(t = t, day = day,
                          covariate = cov_list[[cov_name]])
    cov_mat <- spread(cov_df, key = day, value = covariate) %>%
                 dplyr::select(-t)
    cov_fd  <- smooth.basis(y = as.matrix(cov_mat), fdParobj = BSpline(7))
    cov_pca <- pca.fd(cov_fd$fd, nharm = 6)

    pca_covariates[[spec]][[cov_name]] <- list(
      scores = cov_pca$scores[, 1:2],
      eigen  = cov_pca$values[1:2])
      }
}

# ===== LOOPING DIAGNOSTIK UNTUK 9 MODEL =====

all_results <- list()

for (m in 1:9) {
  spec_m  <- configs$spec[m]
  bw_m    <- configs$bw[m]
  model_name <- paste0("M", m, "_", spec_m, "_bw", bw_m)

  cat("MODEL", m, ":", model_name, "\n")

  # --- 5a. Ambil residual dari Excel ---
  resid_vec <- resid_all[[paste0("Residual_M", m)]]

  # --- 5b. Functional data residual ---
  resid_df  <- data.frame(t = t, day = day, resid = resid_vec)
  resid_mat <- spread(resid_df, key = day, value = resid) %>%
                 dplyr::select(-t)
  resid_fd  <- smooth.basis(y = as.matrix(resid_mat), fdParobj = BSpline(7))

  # --- 5c. PCA residual ---
  resid_pca    <- pca.fd(resid_fd$fd, nharm = 6)
  resid_scores <- resid_pca$scores[, 1:2]
  eigen_resid  <- resid_pca$values[1:2]

  resid_scores_df <- data.frame(
    k     = rep(1:2, each = nrow(resid_scores)),
    resid = as.vector(resid_scores)
  )

  # --- 5d. Loop bin x kovariat ---
  results_model  <- list()
  cov_names_spec <- names(pca_covariates[[spec_m]])

  for (b in names(bins_list)) {
    cat("\n  --- BIN:", b, "---\n")
    results_model[[b]] <- list()

    for (cov_name in cov_names_spec) {
      cov_sc <- pca_covariates[[spec_m]][[cov_name]]$scores
      colnames(cov_sc) <- c("PC1", "PC2")

      scores_temp <- cbind(resid_scores_df, cov_sc)

      results_model[[b]][[cov_name]] <- residTest(
        scores          = scores_temp,
        eigen.resid     = eigen_resid,
        eigen.covariate = pca_covariates[[spec_m]][[cov_name]]$eigen,
        bin             = bins_list[[b]]
      )

      cat(sprintf("  %-8s | Stat: %.6f | p-value: %.4f\n",
                  cov_name,
                  results_model[[b]][[cov_name]]$stat,
                  results_model[[b]][[cov_name]]$pvalue))
    }
  }

  all_results[[model_name]] <- results_model
}

# ===== TABEL RINGKASAN =====

summary_rows <- list()

for (m in 1:9) {
  spec_m     <- configs$spec[m]
  bw_m       <- configs$bw[m]
  model_name <- paste0("M", m, "_", spec_m, "_bw", bw_m)

  for (b in names(bins_list)) {
    for (cov_name in names(all_results[[model_name]][[b]])) {
      r <- all_results[[model_name]][[b]][[cov_name]]
      summary_rows[[length(summary_rows) + 1]] <- data.frame(
        Model     = paste0("M", m),
        Spec      = spec_m,
        Bandwidth = bw_m,
        Bin       = b,
        Covariate = cov_name,
        Stat      = round(r$stat,    6),
        Pvalue    = round(r$pvalue,  4),
        Signif    = ifelse(r$pvalue < 0.05, "*", ""),
        stringsAsFactors = FALSE
      )
    }
  }
}

summary_df <- do.call(rbind, summary_rows)

print(summary_df, row.names = FALSE)

# Ekspor
write_xlsx(summary_df, "Diagnostic_ChiouMuller_9Model_Triweight.xlsx")


>>> PCA kovariat untuk spesifikasi: TanpaDummy 
  Memproses kovariat: Ylag1 ...
  Memproses kovariat: Ylag7 ...
  Memproses kovariat: Ylag8 ...
  Memproses kovariat: Ylag14 ...
  Memproses kovariat: Ylag15 ...

>>> PCA kovariat untuk spesifikasi: DummyLN 
  Memproses kovariat: Ylag1 ...
  Memproses kovariat: Ylag7 ...
  Memproses kovariat: Ylag8 ...
  Memproses kovariat: Ylag14 ...
  Memproses kovariat: Ylag15 ...
  Memproses kovariat: LN ...

>>> PCA kovariat untuk spesifikasi: Dummy3Type 
  Memproses kovariat: Ylag1 ...
  Memproses kovariat: Ylag7 ...
  Memproses kovariat: Ylag8 ...
  Memproses kovariat: Ylag14 ...
  Memproses kovariat: Ylag15 ...
  Memproses kovariat: NS ...
  Memproses kovariat: M ...
  Memproses kovariat: NM ...
MODEL 1 : M1_TanpaDummy_bw0.043 

  --- BIN: bin1 ---
  Ylag1    | Stat: 48318.046886 | p-value: 0.0710
  Ylag7    | Stat: 30221.232066 | p-value: 0.2080
  Ylag8    | Stat: 39049.395099 | p-value: 0.1180
  Ylag14   | Stat: 46952.923022 | p-value: 0.0750
 

**Out of Sampel Accuracy**

In [7]:
# ===== LOAD KOEFISIEN DARI EXCEL =====
coef_file   <- "Estimasi_FRTVCM_Dummy9Model.xlsx"
sheet_names <- excel_sheets(coef_file)

coef_list <- lapply(sheet_names, function(s) read_xlsx(coef_file, sheet = s))
names(coef_list) <- sheet_names

# ===== KONFIGURASI 9 MODEL =====
bws   <- c(0.043, 0.086, 0.128)
specs <- c("TanpaDummy", "DummyLN", "Dummy3Type")

configs <- expand.grid(bw   = bws,
                       spec = specs,
                       KEEP.OUT.ATTRS = FALSE,
                       stringsAsFactors = FALSE)
configs$model_id   <- paste0("M", 1:9)
configs$sheet_name <- sheet_names

# =============================================================
# 2. LAG SEED & DATA AKTUAL OS
# =============================================================

Ylag1_OS  <- data$Y[87601:87648]
Ylag2_OS  <- data$Y[87553:87600]
Ylag3_OS  <- data$Y[87505:87552]
Ylag4_OS  <- data$Y[87457:87504]
Ylag5_OS  <- data$Y[87409:87456]
Ylag6_OS  <- data$Y[87361:87408]
Ylag7_OS  <- data$Y[87313:87360]
Ylag8_OS  <- data$Y[87265:87312]
Ylag9_OS  <- data$Y[87217:87264]
Ylag10_OS <- data$Y[87169:87216]
Ylag11_OS <- data$Y[87121:87168]
Ylag12_OS <- data$Y[87073:87120]
Ylag13_OS <- data$Y[87025:87072]
Ylag14_OS <- data$Y[86977:87024]
Ylag15_OS <- data$Y[86929:86976]

Yact <- data$Y[87649:87984]

LN_OS <- data$NEWLN[87649:87984]
NS_OS <- data$LN_NS[87649:87984]

# ===== LOOPING 9 MODEL =====

all_overall <- list()
all_daily   <- list()
all_yhat    <- list()

for (m in 1:9) {

  spec_m <- configs$spec[m]
  bw_m   <- configs$bw[m]
  sname  <- configs$sheet_name[m]
  mid    <- configs$model_id[m]

  cat("\n>>> Model", m, "|", spec_m, "| bw =", bw_m, "\n")

  cf <- coef_list[[sname]]

  # --- Ambil koefisien ---
  BetaI  <- cf$`(Intercept)`
  Beta1  <- cf$Ylag1
  Beta7  <- cf$Ylag7
  Beta8  <- cf$Ylag8
  Beta14 <- cf$Ylag14
  Beta15 <- cf$Ylag15

  # --- Dummy term D1 & D2 ---
  if (spec_m == "TanpaDummy") {
    DummyD1 <- 0
    DummyD2 <- 0
  } else if (spec_m == "DummyLN") {
    BetaLN  <- cf$LN
    DummyD1 <- BetaLN * LN_OS[1:48]
    DummyD2 <- BetaLN * LN_OS[49:96]
  } else if (spec_m == "Dummy3Type") {
    BetaNS  <- cf$NS
    DummyD1 <- BetaNS * NS_OS[1:48]
    DummyD2 <- BetaNS * NS_OS[49:96]
  }

  # --- Forecast 7 hari (persis pola syntax asli) ---
  YhatD1 <- BetaI + Beta1*Ylag1_OS  + Beta7*Ylag7_OS  + Beta8*Ylag8_OS  + Beta14*Ylag14_OS + Beta15*Ylag15_OS + DummyD1
  YhatD2 <- BetaI + Beta1*YhatD1    + Beta7*Ylag6_OS  + Beta8*Ylag7_OS  + Beta14*Ylag13_OS + Beta15*Ylag14_OS + DummyD2
  YhatD3 <- BetaI + Beta1*YhatD2    + Beta7*Ylag5_OS  + Beta8*Ylag6_OS  + Beta14*Ylag12_OS + Beta15*Ylag13_OS
  YhatD4 <- BetaI + Beta1*YhatD3    + Beta7*Ylag4_OS  + Beta8*Ylag5_OS  + Beta14*Ylag11_OS + Beta15*Ylag12_OS
  YhatD5 <- BetaI + Beta1*YhatD4    + Beta7*Ylag3_OS  + Beta8*Ylag4_OS  + Beta14*Ylag10_OS + Beta15*Ylag11_OS
  YhatD6 <- BetaI + Beta1*YhatD5    + Beta7*Ylag2_OS  + Beta8*Ylag3_OS  + Beta14*Ylag9_OS  + Beta15*Ylag10_OS
  YhatD7 <- BetaI + Beta1*YhatD6    + Beta7*Ylag1_OS  + Beta8*Ylag2_OS  + Beta14*Ylag8_OS  + Beta15*Ylag9_OS

  Yhat      <- c(YhatD1, YhatD2, YhatD3, YhatD4, YhatD5, YhatD6, YhatD7)
  Yhat_list <- list(YhatD1, YhatD2, YhatD3, YhatD4, YhatD5, YhatD6, YhatD7)

  # --- Overall ---
  RMSE  <- sqrt(sum((Yact - Yhat)^2) / 336)
  MAPE  <- sum(abs((Yact - Yhat) / Yact)) / 336 * 100
  SMAPE <- sum(abs(Yact - Yhat) / ((abs(Yact) + abs(Yhat)) / 2)) / 336 * 100

  cat(sprintf("  Overall | RMSE: %.4f | MAPE: %.4f%% | sMAPE: %.4f%%\n",
              RMSE, MAPE, SMAPE))

  all_overall[[m]] <- data.frame(
    Model     = mid,
    Spec      = spec_m,
    Bandwidth = bw_m,
    RMSE      = RMSE,
    MAPE      = MAPE,
    sMAPE     = SMAPE
  )

  # --- Daily ---
  eval_daily <- lapply(seq_len(7), function(d) {
    idx     <- ((d - 1) * 48 + 1):(d * 48)
    act     <- Yact[idx]
    hat     <- Yhat_list[[d]]
    rmse_d  <- sqrt(mean((act - hat)^2))
    mape_d  <- mean(abs((act - hat) / act)) * 100
    smape_d <- mean(abs(act - hat) / ((abs(act) + abs(hat)) / 2)) * 100
    cat(sprintf("  Day %d   | RMSE: %.4f | MAPE: %.4f%% | sMAPE: %.4f%%\n",
                d, rmse_d, mape_d, smape_d))
    data.frame(Model     = mid,
               Spec      = spec_m,
               Bandwidth = bw_m,
               Day       = d,
               RMSE      = rmse_d,
               MAPE      = mape_d,
               sMAPE     = smape_d)
  })

  all_daily[[m]] <- do.call(rbind, eval_daily)

  # --- Simpan Yhat ---
  all_yhat[[mid]] <- data.frame(
    Time = 1:336,
    Day  = rep(1:7, each = 48),
    Yact = Yact,
    Yhat = Yhat
  )
}

# ===== GABUNG & EKSPOR =====

overall_df <- do.call(rbind, all_overall)
daily_df   <- do.call(rbind, all_daily)

cat("\n\n===== OVERALL ACCURACY =====\n")
print(overall_df, row.names = FALSE, digits = 4)

cat("\n\n===== DAILY ACCURACY =====\n")
print(daily_df, row.names = FALSE, digits = 4)

write.csv(overall_df, "Accuracy_Overall_9Model.csv", row.names = FALSE)
write.csv(daily_df,   "Accuracy_Daily_9Model.csv",   row.names = FALSE)

export_list           <- all_yhat
export_list$Overall   <- overall_df
export_list$Daily     <- daily_df
write_xlsx(export_list, "Forecast_OutSample_9Model.xlsx")


>>> Model 1 | TanpaDummy | bw = 0.043 
  Overall | RMSE: 389.1114 | MAPE: 6.0892% | sMAPE: 5.7908%
  Day 1   | RMSE: 419.3163 | MAPE: 8.1941% | sMAPE: 7.8404%
  Day 2   | RMSE: 799.9001 | MAPE: 17.9533% | sMAPE: 16.2977%
  Day 3   | RMSE: 309.4827 | MAPE: 5.7012% | sMAPE: 5.4717%
  Day 4   | RMSE: 154.4575 | MAPE: 2.5784% | sMAPE: 2.5324%
  Day 5   | RMSE: 95.6472 | MAPE: 1.4609% | sMAPE: 1.4705%
  Day 6   | RMSE: 90.7541 | MAPE: 1.2221% | sMAPE: 1.2376%
  Day 7   | RMSE: 327.3623 | MAPE: 5.5142% | sMAPE: 5.6853%

>>> Model 2 | TanpaDummy | bw = 0.086 
  Overall | RMSE: 389.2713 | MAPE: 6.0935% | sMAPE: 5.7948%
  Day 1   | RMSE: 419.4625 | MAPE: 8.1990% | sMAPE: 7.8449%
  Day 2   | RMSE: 800.0206 | MAPE: 17.9621% | sMAPE: 16.3057%
  Day 3   | RMSE: 309.6846 | MAPE: 5.7114% | sMAPE: 5.4814%
  Day 4   | RMSE: 155.3564 | MAPE: 2.5899% | sMAPE: 2.5433%
  Day 5   | RMSE: 96.1231 | MAPE: 1.4680% | sMAPE: 1.4774%
  Day 6   | RMSE: 90.7683 | MAPE: 1.2201% | sMAPE: 1.2355%
  Day 7   | RMSE: 32